## Langgraph快速入门
### - Graph API

` 前置: uv add langgraph langchain`
  
**1.节点 Nodes**


- 模型与工具初始化

In [ ]:
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GLM_API_KEY")
base_url = os.getenv("GLM_BASE_URL")

model = ChatOpenAI(
    model="glm-4.6",
    api_key=api_key,
    base_url=base_url,
    temperature=0
)

messages = [
    (
        "system",
        "You are a helpful assistant that translates English to Chinese. Translate the user sentence.",
    ),
    ("human", "您好，介绍一下你自己"),
]
ai_msg = model.invoke(messages)
ai_msg


AIMessage(content='Bonjour, présentez-vous.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 767, 'prompt_tokens': 26, 'total_tokens': 793, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 759, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 5}}, 'model_provider': 'openai', 'model_name': 'glm-4.6', 'system_fingerprint': None, 'id': '20260314170403aa6b86ce928b4816', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ceb96-8f17-7c83-9664-85dc0c966877-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 26, 'output_tokens': 767, 'total_tokens': 793, 'input_token_details': {'cache_read': 5}, 'output_token_details': {'reasoning': 759}})

In [ ]:

# Define tools
@tool
def multiply(a: int, b: int) -> int:
    """Multiply `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a * b


@tool
def add(a: int, b: int) -> int:
    """Adds `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a + b


@tool
def divide(a: int, b: int) -> float:
    """Divide `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a / b


# Augment the LLM with tools
tools = [add, multiply, divide]
tools_by_name = {tool.name: tool for tool in tools}
model_with_tools = model.bind_tools(tools)

- 状态

In [ ]:
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
import operator

# 记录上下文以及llm的调用次数
# Annotated[list[AnyMessage], operator.add] 每次添加新的消息时都会使用 operator.add 来将新消息添加到现有的消息列表中。
class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    llm_calls: int

- 模型节点

In [ ]:
from langchain.messages import SystemMessage

def llm_call_node(state: dict):
    """LLM decides whether to call a tool or not"""

    return {
        "messages": [
            model_with_tools.invoke(
                [
                    SystemMessage(
                        content="You are a helpful assistant tasked with performing arithmetic on a set of inputs."
                    )
                ]
                + state["messages"]
            )
        ],
        "llm_calls": state.get('llm_calls', 0) + 1
    }

- 工具执行节点

In [ ]:
from langchain.messages import ToolMessage


def tool_node(state: dict):
    """Performs the tool call"""

    result = []
    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    return {"messages": result}

- 条件边

In [ ]:
from typing import Literal
from langgraph.graph import StateGraph, START, END


def should_continue_edge(state: MessagesState) -> Literal["tool_node", END]:
    """Decide if we should continue the loop or stop based upon whether the LLM made a tool call"""

    messages = state["messages"]
    last_message = messages[-1]

    # If the LLM makes a tool call, then perform an action
    if last_message.tool_calls:
        return "tool_node"

    # Otherwise, we stop (reply to the user)
    return END

- 构建

In [ ]:
# Build workflow
agent_builder = StateGraph(MessagesState)

# Add nodes
agent_builder.add_node("llm_call_node", llm_call_node)
agent_builder.add_node("tool_node", tool_node)

# Add edges to connect nodes
agent_builder.add_edge(START, "llm_call_node")
agent_builder.add_conditional_edges(
    "llm_call_node",
    should_continue_edge,
    ["tool_node", END]
)
agent_builder.add_edge("tool_node", "llm_call_node")

# Compile the agent
agent = agent_builder.compile()

# Show the agent
from IPython.display import Image, display
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

# Invoke
from langchain.messages import HumanMessage
messages = [HumanMessage(content="Add 3 and 4.")]
messages = agent.invoke({"messages": messages})
for m in messages["messages"]:
    m.pretty_print()

### - 函数式API
`在函数式 API 中，无需显式定义节点和边，只需在单个函数中编写标准控制流逻辑（循环、条件语句）即可。`

- 导入

In [ ]:
from langgraph.graph import add_messages
from langchain.messages import (
    SystemMessage,
    HumanMessage,
    ToolCall,
)
from langchain_core.messages import BaseMessage
from langgraph.func import entrypoint, task

- 定义模型节点

> **@task** 装饰器将一个函数标记为一个任务，该任务可以作为代理的一部分执行。任务可以在入口函数中同步或异步调用。

In [ ]:
@task
def call_llm(messages: list[BaseMessage]):
    """LLM decides whether to call a tool or not"""
    return model_with_tools.invoke(
        [
            SystemMessage(
                content="You are a helpful assistant tasked with performing arithmetic on a set of inputs."
            )
        ]
        + messages
    )

- 工具执行节点

In [ ]:
@task
def call_tool(tool_call: ToolCall):
    """Performs the tool call"""
    tool = tools_by_name[tool_call["name"]]
    return tool.invoke(tool_call)

- 构建
> 使用 **@entrypoint** 函数构建。

In [ ]:
@entrypoint()
def agent(messages: list[BaseMessage]):
    model_response = call_llm(messages).result()

    while True:
        if not model_response.tool_calls:
            break

        # Execute tools
        tool_result_futures = [
            call_tool(tool_call) for tool_call in model_response.tool_calls
        ]
        tool_results = [fut.result() for fut in tool_result_futures]
        messages = add_messages(messages, [model_response, *tool_results])
        model_response = call_llm(messages).result()

    messages = add_messages(messages, model_response)
    return messages

display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

# Invoke
messages = [HumanMessage(content="Add 3 and 4.")]
for chunk in agent.stream(messages, stream_mode="updates"):
    print(chunk)
    print("\n")
# messages = agent.invoke(messages)
# for m in messages:
#     m.pretty_print()